    #SUGGESTED WORKFLOW
                  STEP1
             Collection of Data
[] Get the balanced quality dataset from TQC/TMD/MP/C2DB/AFLOW for materials with both Electronic and structural properties and that has a label showing this is Trivial or TI and get the data without the label.

[] Train a model using materials data with topological label

[] Test the model and validate it

[] Then use the model to predict and classify materials (TI Vs Trivial)

[] Interpret the model

      STEP 2
Train Baseline models(Tabular and GNNs)


Tabular models(Random Forest and XGBoost)


Random Forest-An ensemle learning method that builds multiple decision trees during training to achieve higher accuracy and stability than a single tree

In [ ]:
# ---------------------------------------------------------------
# Standard Python libraries
# ---------------------------------------------------------------
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
warnings.filterwarnings('ignore')           # suppress non-critical warnings

# ---------------------------------------------------------------
# Machine learning — scikit-learn
# ---------------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, f1_score,
                             roc_auc_score, classification_report,
                             ConfusionMatrixDisplay, confusion_matrix)
from sklearn.preprocessing import label_binarize

# ---------------------------------------------------------------
# XGBoost — gradient boosted trees
# ---------------------------------------------------------------
import xgboost as xgb
from xgboost import XGBClassifier

# ---------------------------------------------------------------
# SHAP — feature interpretability
# ---------------------------------------------------------------
import shap

# ---------------------------------------------------------------
# PyTorch — deep learning framework
# ---------------------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(42)                       # seed for reproducibility

# ---------------------------------------------------------------
# PyTorch Geometric — graph neural networks
# Refs: Fey & Lenssen (2019), PyG docs
# ---------------------------------------------------------------
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import (CGConv,          # Crystal Graph Conv (Xie & Grossman 2018)
                                 GATConv,         # Graph Attention (Velicković 2018)
                                 global_mean_pool,# pooling node → graph
                                 Set2Set)         # advanced pooling for MEGNet

# ---------------------------------------------------------------
# torch_cluster — radius_graph for spatial neighbor search
# ---------------------------------------------------------------
from torch_cluster import radius_graph

# ---------------------------------------------------------------
# Pymatgen — crystal structure parsing from CIF files
# Ref: Ong et al. (2013)
# ---------------------------------------------------------------
from pymatgen.io.cif import CifParser

# ---------------------------------------------------------------
# PySR — symbolic regression
# ---------------------------------------------------------------
from pysr import PySRRegressor

# Set device: use GPU (CUDA) if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("All imports successful!")

[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/pysr/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliapkg/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.12/dist-packages/juliacall/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] Using Julia 1.11.5 at /usr/local/bin/julia
[juliapkg] Using Julia project at /root/.julia/environments/pyjuliapkg
[juliapkg] Writing Project.toml:
           | [deps]
           | SymbolicRegression = "8254be44-1295-4e6a-a16d-46603ac705cb"
           | Serialization = "9e88b42a-f829-5b0c-bbe9-9e923198166b"
           | PythonCall = "6099a3de-0909-46bc-b1f4-468b9a2dfc0d"
           | OpenSSL_jll = "458c3c95-2e84-50aa-8efc-19380b2a3a95"
           | 
           | [compat]
           | SymbolicRegression = "~1.11"
           | Serialization = "^1"
           | PythonCall = "=0.9.26"
           | OpenSSL_jll = "~3.0"
[juliapkg] Installing packages:
           | impo

In [ ]:
# Count samples per class
class_counts = y.value_counts().sort_index()
class_pct    = y.value_counts(normalize=True).sort_index() * 100

print("Class counts:")
print(f"  Trivial (0): {class_counts[0]}  ({class_pct[0]:.1f}%)")
print(f"  Topological (1): {class_counts[1]}  ({class_pct[1]:.1f}%)")

# ---------------------------------------------------------------
# Bar plot of class distribution
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['Trivial (0)', 'Topological (1)'],
       [class_counts[0], class_counts[1]],
       color=['steelblue', 'coral'], edgecolor='black')
ax.set_title('Class Distribution — TI vs Trivial', fontsize=13)
ax.set_ylabel('Number of samples')
for i, v in enumerate([class_counts[0], class_counts[1]]):
    ax.text(i, v + 30, str(v), ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Confusion matrix for Random Forest (Test set)
# ---------------------------------------------------------------
cm_rf = confusion_matrix(y_test, y_test_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=['Trivial', 'Topological'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Random Forest — Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Plot 5: Dependence Plots for top features
# A dependence plot shows how SHAP value for one feature varies
# with its actual value, and how another feature interacts.
# We plot for the top-3 features by mean |SHAP| importance.
# ---------------------------------------------------------------
print("--- Plot 5: SHAP Dependence Plots (Top 3 features) ---")

# Get feature importance (mean absolute SHAP value per feature)
mean_shap = np.abs(sv_rf).mean(axis=0)
top3_idx  = np.argsort(mean_shap)[::-1][:3]   # indices of top 3 features

for idx in top3_idx:
    feat_name = FEATURES[idx]
    plt.figure(figsize=(7, 5))
    shap.dependence_plot(
        ind          = idx,               # which feature to put on x-axis
        shap_values  = sv_rf,
        features     = X_train.values,
        feature_names= FEATURES,
        interaction_index = 'auto',       # auto-detect colour interaction feature
        show         = True
    )
    plt.title(f'Dependence Plot — {feat_name}')
    plt.tight_layout()
    plt.show()

In [ ]:
# ---------------------------------------------------------------
# Global feature columns used in the GNN (11 numeric features)
# NOTE: we use spacegroup_number (int) here, NOT the string spacegroup
# ---------------------------------------------------------------
GLOBAL_FEATURES = [
    'spacegroup_number',   # integer space group (1–230)
    'number of atoms',     # atoms per unit cell
    'Band Gap',            # band gap in eV
    'a', 'b', 'c',         # lattice constants (Angstroms)
    'alpha', 'beta', 'gamma',  # lattice angles (degrees)
    'Z',                   # mean atomic number
    'electronegativity'    # mean Pauling electronegativity
]
N_GLOBAL = len(GLOBAL_FEATURES)   # = 11
print(f"Number of global features: {N_GLOBAL}")

# ---------------------------------------------------------------
# Function: build_graph
# Input:  one row from df_model DataFrame
# Output: torch_geometric.data.Data object
#
# Based on CGCNN approach (Xie & Grossman 2018):
#   - atom node features: Z and electronegativity
#   - edges: all pairs within 5 Å cutoff
#   - edge feature: distance
#   - global: table-level descriptors
# ---------------------------------------------------------------
def build_graph(row, cif_base=CIF_BASE, cutoff=5.0):
    """
    Parse a CIF file and convert to a PyTorch Geometric Data object.

    Args:
        row      : one row of df_model (pandas Series)
        cif_base : base directory where cif_files/ folder lives
        cutoff   : neighbour cutoff radius in Angstrom (default 5.0 Å)

    Returns:
        data : torch_geometric.data.Data with fields
               x        — node features [N_atoms, 2]
               edge_index — COO edge list [2, E]
               edge_attr  — edge features (distances) [E, 1]
               u          — global features [1, 11]
               y          — label [1,]
               pos        — Cartesian coordinates [N_atoms, 3]
    """
    # ---- Parse CIF ----
    # row['cif'] is e.g. 'cif_files/mp-331.cif'
    # We join with cif_base to get the absolute path
    cif_rel  = row['cif']                              # relative path from CSV
    cif_path = os.path.join(cif_base, cif_rel)         # absolute path

    # CifParser reads the .cif file and returns pymatgen Structure objects
    parser    = CifParser(cif_path)
    structure = parser.get_structures()[0]              # get first structure

    # ---- Node features ----
    # For each atomic site in the structure extract:
    #   Z = atomic number (e.g. Si=14, Bi=83)
    #   X = Pauling electronegativity (if missing default to 0)
    atom_Z   = [float(site.specie.Z)
                for site in structure]
    atom_X   = [float(site.specie.X) if site.specie.X is not None else 0.0
                for site in structure]

    # Stack into [N_atoms, 2] tensor
    x = torch.tensor(list(zip(atom_Z, atom_X)), dtype=torch.float)

    # ---- Atomic positions (Cartesian coordinates) ----
    # Used by radius_graph and for attention visualisation
    pos = torch.tensor(structure.cart_coords, dtype=torch.float)  # [N, 3]

    # ---- Edge construction via radius_graph ----
    # radius_graph returns all pairs (i, j) where dist(i,j) < cutoff
    # loop=False excludes self-loops
    edge_index = radius_graph(pos, r=cutoff, loop=False)   # [2, E]

    # ---- Edge features: interatomic distances ----
    src, dst = edge_index                         # source and destination atom indices
    # Euclidean distance between each connected pair
    edge_attr = torch.norm(pos[src] - pos[dst],
                           dim=1, keepdim=True)   # [E, 1]

    # ---- Global features vector u ----
    # 11 numeric descriptors from the DataFrame row
    global_vals = [float(row[c]) for c in GLOBAL_FEATURES]
    u = torch.tensor(global_vals, dtype=torch.float).unsqueeze(0)  # [1, 11]

    # ---- Label ----
    y = torch.tensor([int(row['label'])], dtype=torch.long)   # [1,]

    # ---- Build Data object ----
    data = Data(
        x          = x,           # node features [N, 2]
        edge_index = edge_index,   # edge list     [2, E]
        edge_attr  = edge_attr,    # edge features [E, 1]
        u          = u,            # global feats  [1, 11]
        y          = y,            # label         [1,]
        pos        = pos           # coordinates   [N, 3] (for visualisation)
    )
    return data

# ---- Quick sanity check on first training sample ----
sample = build_graph(df_train.iloc[0])
print("Sample graph:")
print(f"  Atoms (nodes)     : {sample.x.shape}")
print(f"  Edges             : {sample.edge_index.shape}")
print(f"  Edge features     : {sample.edge_attr.shape}")
print(f"  Global features u : {sample.u.shape}")
print(f"  Label y           : {sample.y.item()}")
print(f"  Positions         : {sample.pos.shape}")

In [ ]:
# ---------------------------------------------------------------
# CrystalDataset: wraps a DataFrame of CIF rows into a PyG Dataset
# Inherits from torch_geometric.data.Dataset
# len() returns number of materials
# get(idx) builds the graph for that material on-the-fly
# ---------------------------------------------------------------
class CrystalDataset(Dataset):
    def __init__(self, df, cif_base=CIF_BASE):
        super().__init__()
        self.df       = df.reset_index(drop=True)  # reset index for iloc access
        self.cif_base = cif_base

    def len(self):
        """Return total number of graphs in this split."""
        return len(self.df)

    def get(self, idx):
        """Build and return graph for material at position idx."""
        return build_graph(self.df.iloc[idx], cif_base=self.cif_base)


# ---------------------------------------------------------------
# Build train / val / test datasets
# ---------------------------------------------------------------
print("Building crystal graph datasets... (this may take a few minutes)")

train_dataset = CrystalDataset(df_train)
val_dataset   = CrystalDataset(df_val)
test_dataset  = CrystalDataset(df_test)

print(f"Train graphs : {len(train_dataset)}")
print(f"Val graphs   : {len(val_dataset)}")
print(f"Test graphs  : {len(test_dataset)}")

# ---------------------------------------------------------------
# Create DataLoaders
# DataLoader batches multiple graphs: node features are stacked,
# edge_index is offset per graph, u is stacked [batch, 11]
# ---------------------------------------------------------------
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0)

print(f"\nBatch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

# Check one batch
batch_sample = next(iter(train_loader))
print(f"\nSample batch: {batch_sample}")
print(f"  Batch.x shape   : {batch_sample.x.shape}")      # [total_atoms, 2]
print(f"  Batch.u shape   : {batch_sample.u.shape}")      # [32, 11]
print(f"  Batch.y shape   : {batch_sample.y.shape}")      # [32,]

In [ ]:
# ---------------------------------------------------------------
# CharlesCGCNN — Crystal Graph Convolutional Neural Network
#
# Architecture follows Xie & Grossman (2018):
#   - CGConv layer 1: input (2) → hidden (64)
#   - CGConv layer 2: hidden (64) → hidden (64)
#   - global_mean_pool: aggregate N atoms → 1 graph vector
#   - Concatenate global u (11 dims)
#   - Linear: (64+11) → 2 classes
# ---------------------------------------------------------------
class CharlesCGCNN(nn.Module):
    def __init__(self,
                 node_feat_dim = 2,    # [Z, electronegativity]
                 edge_feat_dim = 1,    # [interatomic distance]
                 hidden_dim    = 64,   # hidden channel width
                 global_dim    = 11,   # length of u vector
                 num_classes   = 2):   # TI vs Trivial
        super().__init__()

        # CGConv layer 1: maps node features from 2-dim to hidden_dim
        # channels=(in, out) defines input and output node feature dims
        # dim=edge_feat_dim tells CGConv the size of edge features
        self.conv1 = CGConv(channels=(node_feat_dim, hidden_dim),
                            dim=edge_feat_dim)

        # CGConv layer 2: keeps node features at hidden_dim
        self.conv2 = CGConv(channels=(hidden_dim, hidden_dim),
                            dim=edge_feat_dim)

        # Batch normalisation after each convolution layer for stable training
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Final classification layer
        # Input = pooled atom embedding (hidden_dim) + global features (global_dim)
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data):
        """
        Forward pass through the Charles CGCNN.

        Args:
            data: batched PyG Data with x, edge_index, edge_attr, batch, u
        Returns:
            logits: [batch_size, 2] — raw scores for each class
        """
        # Unpack graph components
        x         = data.x           # [total_atoms, 2]
        edge_index = data.edge_index  # [2, total_edges]
        edge_attr  = data.edge_attr   # [total_edges, 1]
        batch      = data.batch       # [total_atoms,] — maps atom to graph
        u          = data.u           # [batch_size, 11]

        # ---- Message Passing Layer 1 ----
        # CGConv aggregates neighbour info: each atom gathers info from
        # its bonded neighbours within the 5 Å cutoff
        x = self.conv1(x, edge_index, edge_attr)   # [total_atoms, hidden_dim]
        x = self.bn1(x)
        x = F.relu(x)                               # non-linearity

        # ---- Message Passing Layer 2 ----
        x = self.conv2(x, edge_index, edge_attr)   # [total_atoms, hidden_dim]
        x = self.bn2(x)
        x = F.relu(x)

        # ---- Global Mean Pooling ----
        # Average all atom embeddings in each graph → one vector per crystal
        x = global_mean_pool(x, batch)             # [batch_size, hidden_dim]

        # ---- Concatenate Global Features ----
        # Append lattice/electronic descriptors from the table
        # u has shape [batch_size, 11]
        out = torch.cat([x, u], dim=1)             # [batch_size, hidden_dim+11]

        # ---- Classification ----
        out = self.fc(out)                         # [batch_size, 2]
        return out


# ---- Instantiate the model ----
torch.manual_seed(42)
charles_cgcnn = CharlesCGCNN(
    node_feat_dim = 2,
    edge_feat_dim = 1,
    hidden_dim    = 64,
    global_dim    = N_GLOBAL,   # 11
    num_classes   = 2
).to(device)

print("Charles CGCNN Architecture:")
print(charles_cgcnn)
total_params = sum(p.numel() for p in charles_cgcnn.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

In [ ]:
# ---------------------------------------------------------------
# Training setup
# ---------------------------------------------------------------
cgcnn_optimizer = torch.optim.Adam(
    charles_cgcnn.parameters(),
    lr           = 1e-3,     # learning rate
    weight_decay = 1e-5      # L2 regularisation to prevent overfitting
)
criterion = nn.CrossEntropyLoss()  # combines log-softmax + NLL loss

# Store metrics for plotting later
cgcnn_history = {
    'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []
}

NUM_EPOCHS = 50   # increase to 100+ for full training; 50 for demonstration

print(f"Training CharlesCGCNN for {NUM_EPOCHS} epochs...")
print(f"Device: {device}")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS + 1):

    # ----------------------------------------------------------
    # Training phase
    # ----------------------------------------------------------
    charles_cgcnn.train()    # set to training mode (enables dropout, batchnorm)
    total_train_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)         # move graph to GPU/CPU

        cgcnn_optimizer.zero_grad()      # clear gradients from previous step
        out  = charles_cgcnn(batch)      # forward pass → logits [B, 2]
        loss = criterion(out, batch.y)   # compute cross-entropy loss
        loss.backward()                  # backpropagation
        cgcnn_optimizer.step()           # update weights

        total_train_loss += loss.item() * batch.num_graphs  # accumulate loss

    avg_train_loss = total_train_loss / len(train_dataset)

    # ----------------------------------------------------------
    # Validation phase (no gradient computation needed)
    # ----------------------------------------------------------
    charles_cgcnn.eval()
    val_preds, val_probs, val_labels = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out   = charles_cgcnn(batch)

            # argmax gives predicted class (0 or 1)
            pred  = out.argmax(dim=1).cpu().numpy()
            # softmax gives probability; take column 1 for P(Topological)
            prob  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            lbl   = batch.y.cpu().numpy()

            val_preds.extend(pred)
            val_probs.extend(prob)
            val_labels.extend(lbl)

    # Compute metrics
    acc = accuracy_score(val_labels, val_preds)
    f1  = f1_score(val_labels, val_preds, zero_division=0)
    auc = roc_auc_score(val_labels, val_probs)

    # Store history
    cgcnn_history['train_loss'].append(avg_train_loss)
    cgcnn_history['val_acc'].append(acc)
    cgcnn_history['val_f1'].append(f1)
    cgcnn_history['val_auc'].append(auc)

    # Print every 5 epochs
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS} | "
              f"Loss: {avg_train_loss:.4f} | "
              f"Val Acc: {acc:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}")

print("\nCharles CGCNN training complete!")

In [ ]:
# ---------------------------------------------------------------
# Plot training curves
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cgcnn_history['train_loss'], color='steelblue', label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('CharlesCGCNN — Training Loss'); axes[0].legend()

axes[1].plot(cgcnn_history['val_acc'],  label='Accuracy', color='green')
axes[1].plot(cgcnn_history['val_f1'],   label='F1-score',  color='orange')
axes[1].plot(cgcnn_history['val_auc'],  label='ROC-AUC',   color='red')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('CharlesCGCNN — Validation Metrics')
axes[1].legend(); axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# Final Test evaluation
# ---------------------------------------------------------------
charles_cgcnn.eval()
test_preds_cgcnn, test_probs_cgcnn, test_labels_cgcnn = [], [], []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out   = charles_cgcnn(batch)
        pred  = out.argmax(dim=1).cpu().numpy()
        prob  = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        lbl   = batch.y.cpu().numpy()
        test_preds_cgcnn.extend(pred)
        test_probs_cgcnn.extend(prob)
        test_labels_cgcnn.extend(lbl)

cgcnn_test_acc = accuracy_score(test_labels_cgcnn, test_preds_cgcnn)
cgcnn_test_f1  = f1_score(test_labels_cgcnn, test_preds_cgcnn, zero_division=0)
cgcnn_test_auc = roc_auc_score(test_labels_cgcnn, test_probs_cgcnn)

print("=== CharlesCGCNN — TEST SET RESULTS ===")
print(f"  Accuracy : {cgcnn_test_acc:.4f}")
print(f"  F1-score : {cgcnn_test_f1:.4f}")
print(f"  ROC-AUC  : {cgcnn_test_auc:.4f}")
print(classification_report(test_labels_cgcnn, test_preds_cgcnn,
                            target_names=['Trivial', 'Topological']))

# Save the model weights to Google Drive
save_path = os.path.join(BASE_DIR, 'Charles_CGCNN.pth')
torch.save(charles_cgcnn.state_dict(), save_path)
print(f"\nModel saved to: {save_path}")

In [ ]:
# ---------------------------------------------------------------
# CharlesAttentionGNN — GAT-based model for attention heatmaps
#
# Architecture:
#   - GATConv layer 1: 2-dim input → 64 * 4 heads = 256-dim output
#   - GATConv layer 2: 256-dim input → 64-dim (1 head)
#   - global_mean_pool
#   - Concatenate u (11 dims)
#   - Linear: (64+11) → 2 classes
# ---------------------------------------------------------------
class CharlesAttentionGNN(nn.Module):
    def __init__(self,
                 node_feat_dim = 2,
                 edge_feat_dim = 1,
                 hidden_dim    = 64,
                 heads         = 4,    # number of attention heads
                 global_dim    = 11,
                 num_classes   = 2):
        super().__init__()

        # ---- GAT Layer 1 ----
        # Multi-head attention: each head has hidden_dim channels
        # Output = hidden_dim * heads = 64 * 4 = 256
        # edge_dim=edge_feat_dim enables edge features in attention
        # concat=True concatenates attention heads (default)
        self.gat1 = GATConv(
            in_channels  = node_feat_dim,
            out_channels = hidden_dim,
            heads        = heads,
            edge_dim     = edge_feat_dim,
            concat       = True          # concatenate heads → hidden*heads
        )

        # ---- GAT Layer 2 ----
        # Single head to reduce back to hidden_dim
        # concat=False averages over heads
        self.gat2 = GATConv(
            in_channels  = hidden_dim * heads,  # 256
            out_channels = hidden_dim,           # 64
            heads        = 1,
            concat       = False                 # average (not concatenate)
        )

        # Batch norms for stable training
        self.bn1 = nn.BatchNorm1d(hidden_dim * heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim)

        # Final classification head
        self.fc = nn.Linear(hidden_dim + global_dim, num_classes)

    def forward(self, data, return_attention=False):
        """
        Forward pass. If return_attention=True, also returns
        the edge_index and attention weights from layer 1 (for heatmaps).
        """
        x          = data.x
        edge_index = data.edge_index
        edge_attr  = data.edge_attr
        batch      = data.batch
        u          = data.u

        # ---- GAT Layer 1 ----
        # return_attention_weights='True' makes GATConv return (x, (edge_index, alpha))
        if return_attention:
            x, (att_edge_index, alpha1) = self.gat1(
                x, edge_index, edge_attr,
                return_attention_weights=True   # capture alpha_ij
            )
        else:
            x = self.gat1(x, edge_index, edge_attr)

        x = self.bn1(x)
        x = F.elu(x)    # ELU activation (standard for GAT)

        # ---- GAT Layer 2 ----
        x = self.gat2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)

        # ---- Pooling + Global concat ----
        x   = global_mean_pool(x, batch)
        out = torch.cat([x, u], dim=1)
        out = self.fc(out)

        if return_attention:
            return out, att_edge_index, alpha1
        return out


# ---- Instantiate ----
torch.manual_seed(42)
charles_attn = CharlesAttentionGNN(
    node_feat_dim = 2,
    edge_feat_dim = 1,
    hidden_dim    = 64,
    heads         = 4,
    global_dim    = N_GLOBAL,
    num_classes   = 2
).to(device)

print("CharlesAttentionGNN:")
print(charles_attn)
attn_params = sum(p.numel() for p in charles_attn.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {attn_params:,}")

In [ ]:
# ---------------------------------------------------------------
# Train CharlesAttentionGNN — same loop as CharlesCGCNN
# ---------------------------------------------------------------
attn_optimizer = torch.optim.Adam(
    charles_attn.parameters(), lr=1e-3, weight_decay=1e-5
)

attn_history = {'train_loss': [], 'val_acc': [], 'val_f1': [], 'val_auc': []}

NUM_EPOCHS_ATTN = 50
print(f"Training CharlesAttentionGNN for {NUM_EPOCHS_ATTN} epochs...")
print("-" * 65)

for epoch in range(1, NUM_EPOCHS_ATTN + 1):

    # ---- Training ----
    charles_attn.train()
    total_loss = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        attn_optimizer.zero_grad()
        out  = charles_attn(batch)                    # standard forward (no attention)
        loss = criterion(out, batch.y)
        loss.backward()
        attn_optimizer.step()
        total_loss += loss.item() * batch.num_graphs

    avg_loss = total_loss / len(train_dataset)

    # ---- Validation ----
    charles_attn.eval()
    preds, probs, labels = [], [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(device)
            out   = charles_attn(batch)
            preds.extend(out.argmax(dim=1).cpu().numpy())
            probs.extend(torch.softmax(out, dim=1)[:, 1].cpu().numpy())
            labels.extend(batch.y.cpu().numpy())

    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, zero_division=0)
    auc = roc_auc_score(labels, probs)

    attn_history['train_loss'].append(avg_loss)
    attn_history['val_acc'].append(acc)
    attn_history['val_f1'].append(f1)
    attn_history['val_auc'].append(auc)

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d}/{NUM_EPOCHS_ATTN} | "
              f"Loss: {avg_loss:.4f} | Val Acc: {acc:.4f} | "
              f"F1: {f1:.4f} | AUC: {auc:.4f}")

# ---- Test evaluation ----
charles_attn.eval()
test_preds_attn, test_probs_attn, test_labels_attn = [], [], []
with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        out   = charles_attn(batch)
        test_preds_attn.extend(out.argmax(dim=1).cpu().numpy())
        test_probs_attn.extend(torch.softmax(out, dim=1)[:, 1].cpu().numpy())
        test_labels_attn.extend(batch.y.cpu().numpy())

print("\n=== CharlesAttentionGNN — TEST RESULTS ===")
print(f"  Accuracy : {accuracy_score(test_labels_attn, test_preds_attn):.4f}")
print(f"  F1-score : {f1_score(test_labels_attn, test_preds_attn):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(test_labels_attn, test_probs_attn):.4f}")

# Save model
attn_save = os.path.join(BASE_DIR, 'Charles_AttentionGNN.pth')
torch.save(charles_attn.state_dict(), attn_save)
print(f"Saved to: {attn_save}")